In [2]:
# ----------------------------------------------------------
# STEP 0: IMPORT REQUIRED LIBRARIES
# ----------------------------------------------------------
# PySpark functions:
# Used for data transformation (explode, cleaning, casting)

from pyspark.sql.functions import (
    col,           # Access columns
    explode,       # Convert array -> multiple rows
    trim,          # Remove spaces
    lower,         # Convert text to lowercase
    upper,         # Convert text to uppercase
    to_timestamp   # Convert to timestamp datatype
)

print("All required libraries imported successfully.")

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 3, Finished, Available, Finished, False)

All required libraries imported successfully.


In [3]:
# ----------------------------------------------------------
# STEP 1: READ BRONZE FLATTENED MONGODB TABLE
# ----------------------------------------------------------
# We are reading the Bronze table created from MongoDB ingestion.
# This table already contains flattened customer and payment fields,
# but the products column is still an array.
#
# That means:
# one order still contains multiple products inside one row.
#
# For Silver analytics, we need:
# one row per product.

mongo_bronze_df = spark.table("bronze_mongodb_orders_flat")

print("Bronze MongoDB table loaded successfully.")
display(mongo_bronze_df)

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 4, Finished, Available, Finished, False)

Bronze MongoDB table loaded successfully.


SynapseWidget(Synapse.DataFrame, 14941bc3-64a6-45ba-ab20-61379d008e50)

In [4]:
# ----------------------------------------------------------
# STEP 2: INSPECT THE BRONZE SCHEMA
# ----------------------------------------------------------
# We check the schema to understand the current structure,
# especially the products column.
#
# We expect products to still be an array of structs.

mongo_bronze_df.printSchema()

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 5, Finished, Available, Finished, False)

root
 |-- mongo_id: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: long (nullable = true)
 |-- customer_email: long (nullable = true)
 |-- products: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull = true)
 |-- payment_id: long (nullable = true)
 |-- payment_method: long (nullable = true)
 |-- payment_status: long (nullable = true)
 |-- payment_amount: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_time: timestamp (nullable = true)



In [5]:
# ----------------------------------------------------------
# STEP 3: EXPLODE THE PRODUCTS ARRAY
# ----------------------------------------------------------
# WHY WE DO THIS:
# Bronze has one order containing multiple products in a single row.
#
# Example:
# order_id = 1001
# products = [Laptop, Mouse]
#
# After explode():
# Row 1 -> order_id = 1001, product = Laptop
# Row 2 -> order_id = 1001, product = Mouse
#
# explode() converts each array element into a separate row.

from pyspark.sql.functions import explode, col

mongo_exploded_df = mongo_bronze_df.select(
    col("mongo_id"),
    col("order_id"),
    col("customer_id"),
    col("customer_name"),
    col("customer_email"),
    explode(col("products")).alias("product"),
    col("payment_id"),
    col("payment_method"),
    col("payment_status"),
    col("payment_amount"),
    col("order_status"),
    col("order_time")
)

print("Products array exploded successfully.")
display(mongo_exploded_df)

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 6, Finished, Available, Finished, False)

Products array exploded successfully.


SynapseWidget(Synapse.DataFrame, a0842846-9114-4881-a3d0-15c1b3c4c46e)

In [6]:
# ----------------------------------------------------------
# STEP 4: EXTRACT PRODUCT FIELDS FROM THE EXPLODED STRUCT
# ----------------------------------------------------------
# After explode(), each row has a column called 'product'.
# But product is still a STRUCT, not plain columns.
#
# Example:
# product.product_id
# product.product_name
# product.category
# product.price
#
# We now flatten those nested fields into separate columns.

mongo_silver_base_df = mongo_exploded_df.select(
    col("mongo_id"),
    col("order_id"),
    col("customer_id"),
    col("customer_name"),
    col("customer_email"),
    col("product.product_id").alias("product_id"),
    col("product.product_name").alias("product_name"),
    col("product.category").alias("product_category"),
    col("product.price").alias("product_price"),
    col("payment_id"),
    col("payment_method"),
    col("payment_status"),
    col("payment_amount"),
    col("order_status"),
    col("order_time")
)

print("Product fields extracted successfully.")
display(mongo_silver_base_df)

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 7, Finished, Available, Finished, False)

Product fields extracted successfully.


SynapseWidget(Synapse.DataFrame, 363e546a-0043-44b9-8a23-f71c7307191b)

In [7]:
# ----------------------------------------------------------
# STEP 5: CLEAN AND STANDARDIZE THE DATA
# ----------------------------------------------------------
# We apply basic Silver-layer cleaning:
# 1. trim()         -> remove unwanted spaces
# 2. lower()        -> standardize email and text columns
# 3. to_timestamp() -> ensure order_time is proper timestamp
#
# This makes the dataset cleaner and more consistent.

from pyspark.sql.functions import trim, lower, to_timestamp

mongo_silver_clean_df = mongo_silver_base_df.select(
    col("mongo_id"),
    col("order_id"),
    col("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    lower(trim(col("customer_email"))).alias("customer_email"),
    col("product_id"),
    trim(col("product_name")).alias("product_name"),
    lower(trim(col("product_category"))).alias("product_category"),
    col("product_price"),
    col("payment_id"),
    lower(trim(col("payment_method"))).alias("payment_method"),
    upper(trim(col("payment_status"))).alias("payment_status"),
    col("payment_amount"),
    upper(trim(col("order_status"))).alias("order_status"),
    to_timestamp(col("order_time")).alias("order_time")
)

print("Data cleaned and standardized successfully.")
display(mongo_silver_clean_df)

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 8, Finished, Available, Finished, False)

Data cleaned and standardized successfully.


SynapseWidget(Synapse.DataFrame, 0345d403-ca7c-48a3-bde4-6364f7581d91)

In [8]:
# ----------------------------------------------------------
# STEP 6: REMOVE DUPLICATE ROWS
# ----------------------------------------------------------
# Bronze/Silver pipelines may sometimes ingest duplicate records.
# dropDuplicates() removes exact duplicate rows.
#
# For this learning project, we apply a simple deduplication step.

mongo_silver_final_df = mongo_silver_clean_df.dropDuplicates()

print("Duplicates removed successfully.")
display(mongo_silver_final_df)

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 9, Finished, Available, Finished, False)

Duplicates removed successfully.


SynapseWidget(Synapse.DataFrame, 1ee821ad-c2df-4bf1-b9e7-7f2035ceb55a)

In [9]:
# ----------------------------------------------------------
# STEP 7: INSPECT FINAL SILVER SCHEMA
# ----------------------------------------------------------
# We validate the final Silver schema before writing the table.

mongo_silver_final_df.printSchema()

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 10, Finished, Available, Finished, False)

root
 |-- mongo_id: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: long (nullable = true)
 |-- payment_id: long (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_amount: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_time: timestamp (nullable = true)



In [10]:
# ----------------------------------------------------------
# STEP 8: WRITE THE FINAL SILVER TABLE
# ----------------------------------------------------------
# We now write the cleaned, exploded, analytics-ready output
# to the Silver layer.

mongo_silver_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_mongodb_orders")

print("silver_mongodb_orders created successfully.")

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 11, Finished, Available, Finished, False)

silver_mongodb_orders created successfully.


In [11]:
# ----------------------------------------------------------
# STEP 9: VALIDATE SILVER OUTPUT
# ----------------------------------------------------------
# We query the Silver table and inspect the final result.

spark.sql("""
SELECT *
FROM silver_mongodb_orders
""").show(truncate=False)

StatementMeta(, c03d7623-e1a2-401e-a293-a845a235ec91, 12, Finished, Available, Finished, False)

+------------------------+--------+-----------+-------------+--------------+----------+------------+----------------+-------------+----------+--------------+--------------+--------------+------------+-------------------+
|mongo_id                |order_id|customer_id|customer_name|customer_email|product_id|product_name|product_category|product_price|payment_id|payment_method|payment_status|payment_amount|order_status|order_time         |
+------------------------+--------+-----------+-------------+--------------+----------+------------+----------------+-------------+----------+--------------+--------------+--------------+------------+-------------------+
|69de5c58d5c58a217003c92e|1001    |301        |NULL         |NULL          |3002      |NULL        |NULL            |500          |6001      |NULL          |NULL          |75500         |DELIVERED   |2026-04-14 15:30:00|
|69de5c58d5c58a217003c92e|1001    |301        |NULL         |NULL          |3001      |NULL        |NULL            